In [ ]:
#import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

# EDA

In [ ]:
#read in data file

data = pd.read_csv('stats.csv')

data.head()

In [ ]:
#rows and columns
data.shape

In [ ]:
#no nulls
data.info()

In [ ]:
data.describe()

In [ ]:
#plot home runs

#create bins with 70 being max
max_hr = max(data['home_run'])
bins = range(0, 70, 10)

#plot
fig, ax = plt.subplots(figsize=(10,6))

ax.hist(data['home_run'], bins=bins)
ax.set_xlabel('Home Runs')
ax.set_ylabel('Number of Players')

plt

In [ ]:
#plot at bats (plate attempts)

max_pa = max(data['pa'])
bins = range(0,((max_pa//100) +1) *100, 100) 

#plot
fig, ax = plt.subplots(figsize=(10,6))

ax.hist(data['pa'], bins=bins)
ax.set_xlabel('Plate Attemps')
ax.set_ylabel('Number of Players')

plt

In [ ]:
#plot
fig, ax = plt.subplots(figsize=(10,6))

sns.regplot(data=data, x='pa', y='home_run')

ax.set_xlabel('Plate Appearances')
ax.set_ylabel('Total Home Runs')

plt

In [ ]:
# if we remove any rows with less than 200 plate attempts how much data would that leave me
(data['pa'] >=200).sum()
#data would go from 588 to 348

In [ ]:
data_pa = (data[data['pa'] >=200])
data_pa.head()

In [ ]:
#predictors minus name and year and home_runs 
X = data_pa.drop(columns=['player_id','pa', 'home_run','year', 'last_name, first_name','woba', 'xwoba'])
#target is home runs
y = data_pa['home_run']

#train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=28)

#use defaults for now
rfr = RandomForestRegressor(
    random_state=28,
    n_estimators = 200,
    max_depth = 9
)

#fit model
rfr.fit(X_train, y_train)

#make prediction
y_pred = rfr.predict(X_test)

#check scores
print(rfr.score(X_train, y_train))
print(rfr.score(X_test, y_test))

In [ ]:
#looking at feature importance
importance = rfr.feature_importances_
feature_names = X.columns

#create a dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance}).sort_values(by='importance', ascending=False)

#print results
print(importance_df)

In [ ]:
#save model to use for 2026 data
with open('2025_model.pkl', 'wb') as file:
    pickle.dump(rfr, file)

In [ ]:
#remove woba and xwoba since they favor and weight home runs



In [ ]:
#read in 2026 stats as of 5/27 before any games had been played for the day
data2026 = pd.read_csv('stats2026.csv')

data2026.head()

In [ ]:
#load model
with open('2025_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

In [ ]:
#removing low PA players
bottom25 = np.quantile(data2026['pa'], 0.25)

data2026_pa = (data2026[data2026['pa'] >=bottom25])

#setting X
X_2026 = data2026_pa.drop(columns=['player_id', 'pa', 'home_run','year', 'last_name, first_name','woba', 'xwoba'])

#predict 2026 home runs
data2026_pa['predicted_hrs'] = loaded_model.predict(X_2026)*0.33

In [ ]:
data2026_pa.head()

In [ ]:
#how does this plot?
sns.scatterplot(
    data=data2026_pa,
    x='home_run',
    y='predicted_hrs')


In [ ]:
#determine how accurate the model is
mean_abs_error = mean_absolute_error(data2026_pa['home_run'], data2026_pa['predicted_hrs'])

r2 = r2_score(data2026_pa['home_run'], data2026_pa['predicted_hrs'])

print(f'Mean Absolute Error: {mean_abs_error:.2f} home runs')
print(f'R2 Score: {r2:.2f}')

## Results
### With PA included
- Mean Absolute Error: 3.13 home runs
- R2 Score: -0.13

### Without PA
- Mean Absolute Error: 9.26 home runs
- R2 Score: -8.41

### Multiplying the model by 0.33 to account for on average how much of the 2026 season has been played

- Mean Absolute Error: 2.07 home runs
- R2 Score: 0.42

### PA included, adjusted RFR parameters

- Mean Absolute Error: 3.06 home runs
- R2 Score: -0.09

### PA excluded, adjusted RFR parameters, removed bottom quartile of 2026 data, multiplied by 0.33 to account for season % played

- Mean Absolute Error: 1.91 home runs
- R2 Score: 0.53

In [ ]:
data2026_sorted = data2026_pa.sort_values(by='predicted_hrs', ascending=False)

data2026_sorted.head(10)